# GitHub Analytics with Apilytics

Analyze GitHub issues and PRs using SQL queries.

**Note:** This notebook requires the GitHub config. Start Jupyter with:
```bash
# Dev compose
CATALOG_CONFIG=/opt/spark/examples/github/github-config.conf docker compose up jupyter

# Or dist image
docker run -p 8888:8888 --config /opt/apilytics/configs/github-public.conf neutrinic/apilytics jupyter
```

In [ ]:
# Verify setup
print(f"Spark version: {spark.version}")
print(f"Catalog: {spark.conf.get('spark.sql.catalog.api', 'NOT SET')}")

## Explore Issues

In [ ]:
# List recent issues
spark.sql("""
    SELECT number, title, state, created_at
    FROM api.default.issues
    LIMIT 10
""").show(truncate=40)

In [ ]:
# Check the schema
spark.sql("DESCRIBE api.default.issues").show(truncate=False)

## Filter Pushdown

Filters on configured columns are pushed to the API, reducing data transfer:

In [ ]:
# Filter by state (pushed to API)
spark.sql("""
    SELECT number, title, created_at
    FROM api.default.issues
    WHERE state = 'open'
    LIMIT 10
""").show(truncate=40)

## Issue Analytics

In [ ]:
# Cache issues for analysis
issues = spark.table("api.default.issues").cache()
print(f"Cached {issues.count()} issues")

In [ ]:
# Issues by state
issues.groupBy("state").count().show()

In [ ]:
# Oldest open issues
spark.sql("""
    SELECT number, title, created_at
    FROM api.default.issues
    WHERE state = 'open'
    ORDER BY created_at ASC
    LIMIT 5
""").show(truncate=50)

## Export to Pandas

In [ ]:
# Convert to Pandas for visualization
issues_pd = issues.select("number", "title", "state", "created_at").toPandas()
issues_pd.head()

In [ ]:
# State distribution
issues_pd['state'].value_counts()